def feature_engineering(df):

    df = df.copy()

    # ---------------------------
    # 1. BASIC CLEANING - SORT
    # ---------------------------
    df['dat_izdavanja'] = pd.to_datetime(df['dat_izdavanja'])
    df = df.sort_values(['klijent_id', 'dat_izdavanja'])

    # ---------------------------
    # 1. BASIC CLEANING - HANDLING CATEGORY DATA
    # ---------------------------
    df['property'] = df['property'].astype('category')
    df['gender'] = df['gender'].astype('category')
    df['merrige_status'] = df['merrige_status'].astype('category')
    df['is_foreign'] = df['is_foreign'].astype('category')
    df['property'] = df['property'].astype('category')
    df['birth_place'] = df['birth_place'].astype('category')
    df['osig_mesto_id_y'] = df['osig_mesto_id_y'].astype('category')
    df['osig_opstina_y'] = df['osig_opstina_y'].astype('category')
    df['osig_kanton_y'] = df['osig_kanton_y'].astype('category')
    df['osig_posta_y'] = df['osig_posta_y'].astype('category')
    df['sif_uloga'] = df['sif_uloga'].astype('category')
    df['age_bucket'] = df['age_bucket'].astype('category')
    
    
    # ---------------------------
    # 2. HISTORY FEATURES (ANCHOR LEVEL)
    # ---------------------------
    ## HOW MUCH POLICIES CLIENT HAD BEFORE THIS
    df['n_policies_before'] = df.groupby('klijent_id').cumcount()

    ## HOW MUCH POLICIES CLIENT HAD BEFORE THIS OF THE SAME SIF_VRSTA
    df['cnt_type_before'] = df.groupby(
        ['klijent_id', 'sif_vrsta']
    ).cumcount()

    ## BINARY DOES HE HAVE SAME POLICY BEFORE
    df['had_type_before'] = (df['cnt_type_before'] > 0).astype(int)

    # AVERAGE PREMIUM ON ALL POLICIES BEFORE THIS ONE
    df['avg_premium_past'] = (
        df.groupby('klijent_id')['premija_ukupno']
        .transform(lambda x: x.shift(1).expanding().mean())
    )

    # AVERAGE INSURANCE SUM ON ALL POLICIES BEFORE THIS ONE
    df['avg_insurance_sum'] = (
        df.groupby('klijent_id')['suma_osiguranja']
        .transform(lambda x: x.shift(1).expanding().mean())
    )

    # ---------------------------
    # 3. BUILD SAFE CO-OCCURRENCE (NO LEAKAGE)
    # ---------------------------
    # CREATING DF WITH ONLY THESE 3 COLUMNS
    df_left = df[['klijent_id', 'sif_vrsta', 'dat_izdavanja']].rename(
        columns={'sif_vrsta': 'sif_vrsta_x', 'dat_izdavanja': 't_x'}
    )

    # COPY OF PREVIOUS DF
    df_right = df[['klijent_id', 'sif_vrsta', 'dat_izdavanja']].rename(
        columns={'sif_vrsta': 'sif_vrsta_y', 'dat_izdavanja': 't_y'}
    )

    # CROSS JOIN - MERGING BOOTH DF-S BY KLIJENT_ID SO WE ARE LITERALY GAINING THIS
    # | X      | Y      |
    # | ------ | ------ |
    # | auto   | auto   |
    # | auto   | life   |
    # | auto   | health |
    # | life   | auto   |
    # | life   | life   |
    # | life   | health |
    # | health | auto   |
    # | health | life   |
    # | health | health |
    # EVERY KLIJENT_ID POLICY GETTING ALL KLIJENT_ID POLICIES AS PAIR
    co = df_left.merge(df_right, on='klijent_id')
    # SAME EFFECT LIKE INNER JOIN
    # co = df_left.merge(df_right, on='klijent_id', how='inner')

    # only past relations (NO leakage)
    # FILTER TO NOT INCLUDE POLICIES WHICH ARE CREATED AFTER CURRENT
    # DO NOT COUNT SAME POLICY TYPE
    co = co[co['t_y'] <= co['t_x']]
    co = co[co['sif_vrsta_x'] != co['sif_vrsta_y']]

    # GIVIN PANDAS SERIES LIKE THIS WITH VALUE cnt
    # (auto, life)      120
    # (auto, health)     80
    # (life, auto)       50
    # SO THIS WILL RESET INDEX TO BE
    # | sif_vrsta_x | sif_vrsta_y | cnt |
    # | ----------- | ----------- | --- |
    # | auto        | life        | 120 |
    # | auto        | health      | 80  |
    co = co.groupby(['sif_vrsta_x', 'sif_vrsta_y']).size().reset_index(name='cnt')

    # CALCULATING PROBABILITY
    # | X    | Y      | cnt | prob |
    # | ---- | ------ | --- | ---- |
    # | auto | life   | 80  | 0.8  |
    # | auto | health | 20  | 0.2  |
    co['prob'] = co.groupby('sif_vrsta_x')['cnt'].transform(
        lambda x: x / x.sum()
    )

    # ---------------------------
    # 4. CROSS JOIN (ANCHOR × CANDIDATE)
    # ---------------------------
    # MAKING COPY
    anchors = df.copy()

    # TAKING UNIQUE SIF_VRSTA
    all_types = df['sif_vrsta'].unique()
    # MAKING TABLE OF THIS UNIQUE VALUES
    candidates = pd.DataFrame({'candidate_type': all_types})

    # MAKING WE ARE LOOKING FUTURE POLICY FOR EVERY CLIENT BUYED POLICY
    # SO WHICH POLICIES ARE POSSIBLY BROUGHT NEXT
    dataset = anchors.assign(key=1).merge(
        candidates.assign(key=1),
        on='key'
    ).drop('key', axis=1)

    # ---------------------------
    # 5. LABEL (FUTURE PURCHASE)
    # ---------------------------
    # LITERALY MERGGING ALL CLIENT POLICIES WITH ALL CLIENT POLICIES MAKING EVERY POSIBLE COMBINATION (SQUARE)
    future_pairs = df[['klijent_id', 'sif_vrsta', 'dat_izdavanja']].merge(
        df[['klijent_id', 'sif_vrsta', 'dat_izdavanja']],
        on='klijent_id',
        suffixes=('_x', '_y')
    )

    # LEAVING ONLY FUTURE COMBINATIONS
    future_pairs = future_pairs[
        future_pairs['dat_izdavanja_y'] > future_pairs['dat_izdavanja_x']
    ]

    # REMOVING DUPLICATES
    future_pairs = future_pairs[
        ['klijent_id', 'sif_vrsta_x', 'sif_vrsta_y']
    ].drop_duplicates()

    # MAKING LABEL COLUMN FOR PREDICTIONS
    future_pairs['label'] = 1

    # MERGING WITH ALL UNIQUE POLICY TYPES COMBINATIONS TO MAKE LABEL 1 ONLY ON BROUGHT POLICIES
    dataset = dataset.merge(
        future_pairs,
        left_on=['klijent_id', 'sif_vrsta', 'candidate_type'],
        right_on=['klijent_id', 'sif_vrsta_x', 'sif_vrsta_y'],
        how='left'
    )

    # FILL NOT BROUGHT WITH 0
    dataset['label'] = dataset['label'].fillna(0).astype(int)

    # ---------------------------
    # 6. BASIC FEATURES
    # ---------------------------
    dataset['anchor_type'] = dataset['sif_vrsta']

    # optional but usually good for ranking
    dataset['same_as_anchor'] = (
        dataset['candidate_type'] == dataset['anchor_type']
    ).astype(int)

    # ---------------------------
    # 7. CANDIDATE HISTORY FEATURES
    # ---------------------------
    dataset = dataset.merge(
        df[['klijent_id', 'sif_vrsta', 'cnt_type_before']],
        left_on=['klijent_id', 'candidate_type'],
        right_on=['klijent_id', 'sif_vrsta'],
        how='left'
    )

    dataset['cnt_type_before'] = dataset['cnt_type_before'].fillna(0)

    # ---------------------------
    # 8. DEMOGRAPHICS
    # ---------------------------
    # dataset['property'] = dataset['property'].astype('category')

    # ---------------------------
    # 9. INTERACTIONS
    # ---------------------------
    dataset['property_x_candidate'] = (
        dataset['property'].astype(str) + "_" +
        dataset['candidate_type'].astype(str)
    )

    dataset['candidate_is_new'] = (
        dataset['cnt_type_before'] == 0
    ).astype(int)

    # ---------------------------
    # 10. BAYESIAN FEATURE (FROM SAFE CO-OCCURRENCE)
    # ---------------------------
    dataset = dataset.merge(
        co,
        left_on=['anchor_type', 'candidate_type'],
        right_on=['sif_vrsta_x', 'sif_vrsta_y'],
        how='left'
    )

    dataset['bayes_score'] = dataset['prob'].fillna(0)


    # ---------------------------
    # 11. FINAL FEATURE SET
    # ---------------------------
    features = [
        # core
        'n_policies_before',
        'had_type_before',
        'cnt_type_before',
        'same_as_anchor',
        'bayes_score',

        # history
        'avg_premium_past',

        # demografija
        'age',
        'gender',
        'property',
        'merrige_status',
        'is_foreign',
        'birth_place',
        'osig_mesto_id_y',
        'osig_opstina_y',
        'osig_opstina_y',
        'osig_kanton_y',
        'osig_posta_y',
        'sif_uloga',
        'age_bucket',

        # interaction
        'candidate_is_new',
        'property_x_candidate'
    ]

    return dataset, features